# 01 — Setup & Verification (Gemma 3 4B IT)

Verifies the full stack works on Colab A100 before running real experiments:
1. MGSM loads for all 5 languages
2. Gemma 3 4B IT loads in BF16
3. Gemma Scope 2 SAE loads via SAELens
4. nnsight extracts residual-stream activations
5. 5-problem baseline accuracy is sane

**Run on Colab Pro+ A100.** All subsequent notebooks (02–06) depend on this one passing.

## 0. Clone repo and install

In [ ]:
import os
from pathlib import Path

# Clone the repo into Colab if not already present, then chdir into it.
REPO_URL = 'https://github.com/kvrancic/nlp-project.git'
REPO_DIR = '/content/nlp-project'

if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f'Repo already at {REPO_DIR}, pulling latest')
    !cd {REPO_DIR} && git pull --ff-only

os.chdir(REPO_DIR)
print('CWD:', os.getcwd())

In [ ]:
# Install dependencies. Pinning numpy>=2 to avoid the binary-incompat issue
# the friend hit on Colab. -e installs the local src/ package so we can
# `from src import ...` from anywhere in this kernel.
!pip install -q 'numpy>=2.0' -e .

In [ ]:
# Pull HF token from Colab secrets and write it to .env so src.model picks it up.
from google.colab import userdata

token = userdata.get('HF_TOKEN')
assert token, 'Add HF_TOKEN to Colab secrets (left sidebar key icon).'
os.environ['HF_TOKEN'] = token
Path('.env').write_text(f'HF_TOKEN={token}\n')
print('HF_TOKEN configured')

In [ ]:
import torch

print(f'PyTorch:  {torch.__version__}')
print(f'CUDA:     {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU:      {props.name}')
    print(f'VRAM:     {props.total_memory / 1e9:.1f} GB')
    assert 'A100' in props.name or 'H100' in props.name or 'L4' in props.name, \
        f'Expected A100/H100/L4, got {props.name}. Switch runtime → A100 GPU.'

## 1. Load MGSM

In [ ]:
from src.data import load_mgsm
from src.config import TARGET_LANGUAGES

mgsm = load_mgsm(TARGET_LANGUAGES)
for lang in TARGET_LANGUAGES:
    n = len(mgsm[lang])
    ex = mgsm[lang][0]
    print(f'{lang}: {n} examples | Q[:80]={ex["question"][:80]!r} | A={ex["answer_number"]}')

# Sanity: same problem index → same gold across languages
golds = {lang: mgsm[lang][0]['answer_number'] for lang in TARGET_LANGUAGES}
assert len(set(golds.values())) == 1, f'Problem 0 gold differs across langs: {golds}'
print(f'\nProblem 0 gold consistent across all 5 languages: {next(iter(golds.values()))}')

## 2. Load Gemma 3 4B IT

In [ ]:
from src.model import load_model_and_tokenizer

model, tokenizer = load_model_and_tokenizer()

n_params = sum(p.numel() for p in model.parameters())
n_layers = len(model.model.layers)
d_model = model.config.hidden_size
print(f'Params:    {n_params/1e9:.2f}B')
print(f'Layers:    {n_layers}')
print(f'd_model:   {d_model}')
print(f'Device:    {next(model.parameters()).device}')
print(f'Dtype:     {next(model.parameters()).dtype}')

# Hard-fail if architecture doesn't match what config.py expects.
from src.config import N_LAYERS, D_MODEL
assert n_layers == N_LAYERS, f'Expected {N_LAYERS} layers, got {n_layers}'
assert d_model == D_MODEL, f'Expected d_model={D_MODEL}, got {d_model}'

In [ ]:
# Sanity-generate on a single English problem.
test = mgsm['en'][0]
prompt = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': test['question']}],
    tokenize=False,
    add_generation_prompt=True,
)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    gen = model.generate(**inputs, max_new_tokens=512, do_sample=False)
out = tokenizer.decode(gen[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

from src.data import parse_answer_number
parsed = parse_answer_number(out)
print(f'Q: {test["question"][:120]}...')
print(f'\nGenerated:\n{out}\n')
print(f'Parsed: {parsed} | Gold: {test["answer_number"]} | Correct: {parsed == test["answer_number"]}')

## 3. Load a Gemma Scope 2 SAE

SAELens release: `gemma-scope-2-4b-it-res`. sae_id format: `layer_{N}_width_{W}_l0_{SIZE}`.
Subset layers {9,17,22,29}: widths 16k/65k/262k/1m, L0 small/medium/big.
All-layer release `gemma-scope-2-4b-it-res-all` covers layers 0–33 at widths 16k/262k, L0 small/big.

In [ ]:
from src.model import load_sae
from src.config import SAE_WIDTH_16K

# Smallest combo first to verify the loader works at all.
sae, cfg, sparsity = load_sae(layer=9, width=SAE_WIDTH_16K, l0_target='medium')
print(f'SAE loaded')
print(f'  d_in:    {sae.cfg.d_in}')
print(f'  d_sae:   {sae.cfg.d_sae}')
print(f'  W_enc:   {sae.W_enc.shape}')
print(f'  W_dec:   {sae.W_dec.shape}')
print(f'  device:  {sae.W_dec.device}')

In [ ]:
# Encode/decode test on a synthetic activation.
dummy = torch.randn(4, sae.cfg.d_in, device='cuda', dtype=torch.float32)
with torch.no_grad():
    feats = sae.encode(dummy)
    recon = sae.decode(feats)
n_active = (feats > 0).float().sum(dim=-1).mean().item()
mse = ((dummy - recon) ** 2).mean().item()
print(f'Features:    {feats.shape}')
print(f'Active /tok: {n_active:.1f} / {sae.cfg.d_sae}  ({100*n_active/sae.cfg.d_sae:.2f}%)')
print(f'Recon MSE:   {mse:.4f}  (high here is expected — random input is OOD)')

## 4. nnsight residual-stream extraction

In [ ]:
from src.extraction import extract_residual_activations, encode_activations_through_sae
from src.config import SAE_SUBSET_LAYERS

sample_texts = [mgsm['en'][i]['question'] for i in range(4)]
acts = extract_residual_activations(
    model, tokenizer, sample_texts,
    layers=SAE_SUBSET_LAYERS,
    batch_size=4,
    positions='last',
)
for layer in SAE_SUBSET_LAYERS:
    print(f'  layer {layer:>2}: {tuple(acts[layer].shape)}  norm_mean={acts[layer].float().norm(dim=-1).mean():.2f}')

In [ ]:
# Encode the layer-9 activations through the loaded SAE.
feats_l9 = encode_activations_through_sae(acts[9], sae)
print(f'SAE features (layer 9): {tuple(feats_l9.shape)}')
print(f'Active features per ex: {(feats_l9 > 0).float().sum(dim=-1).tolist()}')
print(f'Top-5 features for ex 0: {torch.topk(feats_l9[0], k=5).indices.tolist()}')

## 5. 5-problem baseline accuracy

In [ ]:
from src.data import format_prompt_gemma_it
from src.evaluation import evaluate_mgsm

N_TEST = 5
results = {}
for lang in TARGET_LANGUAGES:
    qs = [mgsm[lang][i]['question'] for i in range(N_TEST)]
    golds = [mgsm[lang][i]['answer_number'] for i in range(N_TEST)]
    # Use the chat template via the tokenizer for production-quality prompts.
    prompts = [
        tokenizer.apply_chat_template(
            [{'role': 'user', 'content': q}],
            tokenize=False, add_generation_prompt=True,
        )
        for q in qs
    ]
    res = evaluate_mgsm(model, tokenizer, prompts, golds, max_new_tokens=512)
    results[lang] = res
    print(f'{lang}: {res["accuracy"]:.0%} ({sum(res["correct"])}/{N_TEST})')

avg = sum(r['accuracy'] for r in results.values()) / len(results)
print(f'\nAverage across 5 langs: {avg:.0%}')

## 6. Memory check

In [ ]:
if torch.cuda.is_available():
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    alloc = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f'Allocated: {alloc:5.2f} GB')
    print(f'Reserved:  {reserved:5.2f} GB')
    print(f'Total:     {total:5.2f} GB')
    print(f'Free:      {total - alloc:5.2f} GB  (need ~5GB headroom for 4× SAEs in phase 1)')

## Summary

If every cell ran without error and the 5-problem baseline showed sensible numbers (English typically ≥60%, Bengali/Swahili lower), the stack is verified.

**Report back:**
- Per-language baseline accuracies
- The exact `sae_id` that loaded successfully
- VRAM used after loading model + 1 SAE
- Any errors hit

After this, proceed to `02_feature_extraction.ipynb`.